# 🦴 Diz Osteoartrit (KL Grading) Sınıflandırması

## Sinir Ağları Dersi - Bitirme Ödevi

Bu notebook, Kellgren-Lawrence (KL) skalasına göre diz röntgen görüntülerini 5 sınıfa (Grade 0-4) ayıran bir derin öğrenme modeli içermektedir.

### İçindekiler
1. Kütüphanelerin Yüklenmesi
2. Veri Setinin Hazırlanması
3. Veri Görselleştirme
4. Model Mimarisi (Özgün CNN)
5. Transfer Learning (ResNet18)
6. Model Eğitimi
7. Değerlendirme ve Sonuçlar

---
## 1. Kütüphanelerin Yüklenmesi

In [ ]:
# Temel kütüphaneler
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Sklearn metrikleri
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Görselleştirme ayarları
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Cihaz kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan cihaz: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Tekrarlanabilirlik için seed ayarla
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

---
## 2. Veri Setinin Hazırlanması

### 2.1 Veri Dönüşümleri (Transforms)

- **Eğitim seti**: Veri artırma teknikleri (augmentation) uygulanır
- **Validasyon/Test seti**: Sadece resize ve normalize

In [ ]:
# Görüntü boyutu
IMG_SIZE = 224

# ImageNet normalizasyon değerleri
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Eğitim için transforms (veri artırma dahil)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),        # Yatay çevirme
    transforms.RandomRotation(degrees=10),          # Rastgele döndürme
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Renk değişimi
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Validasyon ve test için transforms
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("Transforms tanımlandı:")
print(f"\nEğitim transforms:\n{train_transform}")
print(f"\nVal/Test transforms:\n{val_test_transform}")

### 2.2 ImageFolder ile Veri Yükleme

In [ ]:
# Veri dizinleri
DATA_DIR = 'data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# ImageFolder ile veri setlerini yükle
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transform)

# Sınıf isimleri
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Sınıflar: {class_names}")
print(f"Sınıf sayısı: {num_classes}")
print(f"\nVeri seti boyutları:")
print(f"  Eğitim: {len(train_dataset)} görüntü")
print(f"  Validasyon: {len(val_dataset)} görüntü")
print(f"  Test: {len(test_dataset)} görüntü")

### 2.3 DataLoader Oluşturma

In [ ]:
# Hiperparametreler
BATCH_SIZE = 32
NUM_WORKERS = 0  # Windows için 0, Linux için artırılabilir

# DataLoader'ları oluştur
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"DataLoader'lar oluşturuldu:")
print(f"  Batch boyutu: {BATCH_SIZE}")
print(f"  Eğitim batch sayısı: {len(train_loader)}")
print(f"  Validasyon batch sayısı: {len(val_loader)}")
print(f"  Test batch sayısı: {len(test_loader)}")

---
## 3. Veri Görselleştirme

### 3.1 Sınıf Dağılımı

In [ ]:
# Her sınıftaki örnek sayısını hesapla
class_counts = [0] * num_classes
for _, label in train_dataset.samples:
    class_counts[label] += 1

# Görselleştirme
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, num_classes))
bars = ax.bar(class_names, class_counts, color=colors, edgecolor='black')

ax.set_xlabel('Sınıf (KL Grade)', fontsize=12)
ax.set_ylabel('Örnek Sayısı', fontsize=12)
ax.set_title('Eğitim Seti Sınıf Dağılımı', fontsize=14, fontweight='bold')

# Bar üzerinde değerleri göster
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nSınıf dağılımı:")
for name, count in zip(class_names, class_counts):
    print(f"  Grade {name}: {count} örnek ({count/len(train_dataset)*100:.1f}%)")

### 3.2 Örnek Görüntüler

In [ ]:
def denormalize(tensor):
    """Normalize edilmiş tensörü görüntüye dönüştür"""
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return tensor * std + mean

# Bir batch al
images, labels = next(iter(train_loader))

# 16 örnek görselleştir
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

for i in range(16):
    img = denormalize(images[i])
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'Grade {class_names[labels[i]]}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Eğitim Setinden Örnek Görüntüler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Model Mimarisi (Özgün CNN)

4 evrişim katmanlı, BatchNorm ve Dropout içeren özel tasarlanmış bir CNN modeli.

In [ ]:
class KneeOsteoarthritisCNN(nn.Module):
    """
    Özgün CNN Mimarisi
    
    Mimari:
        Conv1: 3 → 32 kanal, 3x3 kernel
        Conv2: 32 → 64 kanal, 3x3 kernel
        Conv3: 64 → 128 kanal, 3x3 kernel
        Conv4: 128 → 256 kanal, 3x3 kernel
        FC1: 256 → 128
        FC2: 128 → 5 (sınıf sayısı)
    """
    
    def __init__(self, num_classes=5, dropout_rate=0.5):
        super(KneeOsteoarthritisCNN, self).__init__()
        
        # Evrişim Bloğu 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Evrişim Bloğu 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Evrişim Bloğu 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Evrişim Bloğu 4
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Global Average Pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Tam Bağlantılı Katmanlar
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate * 0.6)
        
    def forward(self, x):
        # Conv Block 1: 224x224 → 112x112
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        
        # Conv Block 2: 112x112 → 56x56
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        
        # Conv Block 3: 56x56 → 28x28
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Conv Block 4: 28x28 → 14x14
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        
        # Global Average Pooling: 14x14 → 1x1
        x = self.global_avg_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC Layers
        x = self.dropout1(x)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        
        return x

In [ ]:
# Modeli oluştur ve test et
cnn_model = KneeOsteoarthritisCNN(num_classes=num_classes)
cnn_model = cnn_model.to(device)

# Parametre sayısı
total_params = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)

print("=" * 60)
print("ÖZGÜN CNN MODELİ")
print("=" * 60)
print(cnn_model)
print("\n" + "-" * 60)
print(f"Toplam parametre: {total_params:,}")
print(f"Eğitilebilir parametre: {trainable_params:,}")

# Test girdisi
dummy_input = torch.randn(1, 3, 224, 224).to(device)
output = cnn_model(dummy_input)
print(f"\nGiriş şekli: {dummy_input.shape}")
print(f"Çıkış şekli: {output.shape}")

---
## 5. Transfer Learning (ResNet18)

ImageNet üzerinde önceden eğitilmiş ResNet18 modelini kullanarak transfer learning.

In [ ]:
class ResNet18TransferModel(nn.Module):
    """
    Transfer Learning Modeli (ResNet18)
    """
    
    def __init__(self, num_classes=5, pretrained=True, freeze_features=False):
        super(ResNet18TransferModel, self).__init__()
        
        # Önceden eğitilmiş ResNet18 yükle
        if pretrained:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
            self.resnet = models.resnet18(weights=weights)
            print("✓ ResNet18 ImageNet ağırlıkları yüklendi.")
        else:
            self.resnet = models.resnet18(weights=None)
        
        # Özellik katmanlarını dondur
        if freeze_features:
            for param in self.resnet.parameters():
                param.requires_grad = False
            print("✓ Özellik çıkarıcı katmanlar donduruldu.")
        
        # Son FC katmanını değiştir
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.resnet(x)

In [ ]:
# ResNet18 modelini oluştur
resnet_model = ResNet18TransferModel(num_classes=num_classes, pretrained=True)
resnet_model = resnet_model.to(device)

# Parametre sayısı
total_params = sum(p.numel() for p in resnet_model.parameters())
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("RESNET18 TRANSFER LEARNING MODELİ")
print("=" * 60)
print(f"Toplam parametre: {total_params:,}")
print(f"Eğitilebilir parametre: {trainable_params:,}")

# Test
output = resnet_model(dummy_input)
print(f"\nGiriş şekli: {dummy_input.shape}")
print(f"Çıkış şekli: {output.shape}")

---
## 6. Model Eğitimi

### 6.1 Eğitim Fonksiyonları

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Bir epoch eğitim"""
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    
    for inputs, labels in tqdm(dataloader, desc="Eğitim", leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = running_corrects.double() / total
    
    return epoch_loss, epoch_acc.item()


def validate(model, dataloader, criterion, device):
    """Validasyon"""
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validasyon", leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = running_corrects.double() / total
    
    return epoch_loss, epoch_acc.item()

### 6.2 Eğitim Döngüsü

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=25, lr=0.001):
    """Ana eğitim fonksiyonu"""
    
    # Loss fonksiyonu: CrossEntropyLoss
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer: Adam
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    
    # Eğitim geçmişi
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_acc = 0.0
    best_model_wts = None
    
    print("=" * 60)
    print("EĞİTİM BAŞLIYOR")
    print("=" * 60)
    print(f"Epoch sayısı: {num_epochs}")
    print(f"Learning rate: {lr}")
    print(f"Optimizer: Adam")
    print(f"Loss: CrossEntropyLoss")
    print("=" * 60 + "\n")
    
    for epoch in range(num_epochs):
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        
        # Eğitim
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validasyon
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )
        
        # Scheduler güncelle
        scheduler.step()
        
        # Geçmişe kaydet
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Sonuçları yazdır
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
        
        # En iyi modeli kaydet
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = model.state_dict().copy()
            print(f"  ✓ En iyi model güncellendi! (Val Acc: {val_acc*100:.2f}%)")
        
        print()
    
    # En iyi ağırlıkları yükle
    if best_model_wts is not None:
        model.load_state_dict(best_model_wts)
    
    print("=" * 60)
    print(f"Eğitim tamamlandı! En iyi Val Acc: {best_acc*100:.2f}%")
    print("=" * 60)
    
    return model, history

### 6.3 Model Eğitimi (Özgün CNN)

In [ ]:
# Özgün CNN modelini eğit
# NOT: Bu hücreyi çalıştırmak birkaç dakika sürebilir

NUM_EPOCHS = 15  # Epoch sayısını ihtiyaca göre ayarlayın
LEARNING_RATE = 0.001

# Modeli yeniden başlat
cnn_model = KneeOsteoarthritisCNN(num_classes=num_classes).to(device)

# Eğit
cnn_model, cnn_history = train_model(
    cnn_model, 
    train_loader, 
    val_loader, 
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE
)

### 6.4 Eğitim Grafikleri

In [ ]:
def plot_training_history(history):
    """Eğitim grafiklerini çiz"""
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss grafiği
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Eğitim Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validasyon Loss', linewidth=2)
    axes[0].set_title('Loss Değişimi', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy grafiği
    axes[1].plot(epochs, [acc*100 for acc in history['train_acc']], 'b-', 
                 label='Eğitim Accuracy', linewidth=2)
    axes[1].plot(epochs, [acc*100 for acc in history['val_acc']], 'r-', 
                 label='Validasyon Accuracy', linewidth=2)
    axes[1].set_title('Accuracy Değişimi', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Grafikleri çiz
plot_training_history(cnn_history)

---
## 7. Değerlendirme ve Sonuçlar

### 7.1 Test Seti Değerlendirmesi

In [ ]:
def evaluate_on_test(model, test_loader, device):
    """Test seti üzerinde değerlendirme"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Test değerlendirmesi"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

# Test değerlendirmesi
y_pred, y_true = evaluate_on_test(cnn_model, test_loader, device)

# Genel doğruluk
test_accuracy = accuracy_score(y_true, y_pred)
print(f"\nTest Seti Doğruluğu: {test_accuracy*100:.2f}%")

### 7.2 Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Normalize edilmiş confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ham değerler
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Grade {c}' for c in class_names],
            yticklabels=[f'Grade {c}' for c in class_names],
            ax=axes[0], square=True)
axes[0].set_title('Confusion Matrix (Sayılar)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Gerçek Etiket')
axes[0].set_xlabel('Tahmin Edilen')

# Normalize edilmiş
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'Grade {c}' for c in class_names],
            yticklabels=[f'Grade {c}' for c in class_names],
            ax=axes[1], square=True)
axes[1].set_title('Confusion Matrix (Normalize)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Gerçek Etiket')
axes[1].set_xlabel('Tahmin Edilen')

plt.tight_layout()
plt.show()

### 7.3 Classification Report (Precision, Recall, F1-Score)

In [ ]:
# Classification Report
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

# Detaylı rapor
report = classification_report(
    y_true, y_pred,
    target_names=[f'Grade {c}' for c in class_names],
    digits=4
)
print(report)

print("=" * 70)
print(f"Genel Doğruluk (Accuracy): {test_accuracy*100:.2f}%")
print("=" * 70)

### 7.4 Sınıf Bazlı Başarı Analizi

In [ ]:
# Her sınıf için doğruluk
print("\nSınıf Bazlı Doğruluk:")
print("-" * 40)
for i, class_name in enumerate(class_names):
    mask = y_true == i
    if mask.sum() > 0:
        class_acc = (y_pred[mask] == y_true[mask]).mean()
        print(f"  Grade {class_name}: {class_acc*100:.2f}% ({mask.sum()} örnek)")

### 7.5 Modeli Kaydetme

In [ ]:
# Model kaydet
os.makedirs('checkpoints', exist_ok=True)

torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'class_names': class_names,
    'accuracy': test_accuracy
}, 'checkpoints/cnn_model.pth')

print("Model kaydedildi: checkpoints/cnn_model.pth")

---
## 8. Transfer Learning ile Eğitim (Opsiyonel)

ResNet18 ile transfer learning kullanarak daha iyi sonuçlar elde edebilirsiniz.

In [ ]:
# ResNet18 Transfer Learning Eğitimi
# NOT: Bu hücreyi çalıştırmak birkaç dakika sürebilir

# Modeli yeniden başlat
resnet_model = ResNet18TransferModel(
    num_classes=num_classes, 
    pretrained=True, 
    freeze_features=False  # Tüm katmanları eğit (fine-tuning)
).to(device)

# Eğit (daha düşük learning rate ile)
resnet_model, resnet_history = train_model(
    resnet_model, 
    train_loader, 
    val_loader, 
    num_epochs=10,
    lr=0.0001  # Transfer learning için düşük LR
)

In [ ]:
# ResNet18 değerlendirmesi
y_pred_resnet, y_true_resnet = evaluate_on_test(resnet_model, test_loader, device)
resnet_accuracy = accuracy_score(y_true_resnet, y_pred_resnet)

print(f"\nResNet18 Test Doğruluğu: {resnet_accuracy*100:.2f}%")
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (ResNet18)")
print("=" * 70)
print(classification_report(
    y_true_resnet, y_pred_resnet,
    target_names=[f'Grade {c}' for c in class_names],
    digits=4
))

---
## 9. Sonuç

Bu projede:

1. **Veri Yükleme**: `torchvision.datasets.ImageFolder` ve `DataLoader` kullanarak train/val/test verileri yüklendi.

2. **Ön İşleme**: 224x224 boyutlandırma, normalizasyon ve veri artırma (RandomHorizontalFlip, RandomRotation, ColorJitter) uygulandı.

3. **Özgün CNN**: 4 evrişim katmanlı, BatchNorm ve Dropout içeren bir model tasarlandı.

4. **Transfer Learning**: ResNet18 ile önceden eğitilmiş ağırlıklar kullanıldı.

5. **Eğitim**: CrossEntropyLoss ve Adam optimizer ile model eğitildi.

6. **Değerlendirme**: Confusion Matrix ve Classification Report ile model başarısı ölçüldü.

### Önemli Notlar

- GPU kullanımı eğitim süresini önemli ölçüde kısaltır.
- Veri seti dengesiz olduğundan, class weights kullanılabilir.
- Transfer Learning genellikle daha iyi sonuçlar verir.
- Epoch sayısı ve learning rate hiperparametre optimizasyonu ile iyileştirilebilir.